# Chapter 03: Raster Images

Source span: *Fundamentals of Computer Graphics*, Chapter 03, printed pp. 63-78, physical PDF pp. 80-95.

## Chapter Goal

A raster image is easy to describe as an array, but most bugs in graphics come from forgetting what the array is standing in for. This chapter builds a working mental model for raster devices, sampled pixels, RGB values, transfer curves, quantized storage, and alpha compositing. The goal is to treat an image array as a measurable object: it has shape, coordinate conventions, value ranges, color encoding, and compositing rules that can be checked.

The source chapter moves from real devices to an abstract image model and then to the pixel formats and operations that a renderer needs. This notebook follows the same conceptual path without copying the book's figures or prose. Every image below is synthetic and generated by code in the notebook.

## Translation guide

| Chapter idea | Computational object in this notebook | What to inspect |
| --- | --- | --- |
| Raster output device | A consumer of an image array: RGB subpixels for displays and binary dots for simple printers | The same array can drive different physical patterns, so an image is device-independent until a device interprets it. |
| Raster input device | A producer of samples: a Bayer-style mosaic records one color component per sensor cell | The raw measurements are not always complete RGB pixels. Software fills in missing channels later. |
| Pixel coordinate convention | Integer sample centers with image domain extending half a pixel beyond the first and last centers | The pixel at array coordinate `(row, column)` is a local sample, not a tiny square that defines all geometry. |
| Pixel value | A scalar or vector stored with finite precision | Shape, channel count, dtype, min, max, and memory footprint are part of the image description. |
| RGB color | A point in the unit cube `[0, 1]^3`, usually stored after quantization | Additive primaries make yellow, cyan, magenta, and white at cube corners. |
| Monitor gamma | A transfer curve between stored code value and emitted light intensity | Linear light calculations and encoded display values are different spaces. |
| Quantization and clipping | Rounding to finite levels and clamping values outside the display range | Banding and saturated highlights are storage artifacts, not scene properties. |
| Alpha compositing | Weighted coverage/opacity, preferably stored as premultiplied color when filtering or interpolation is involved | The invariant is the over operation: foreground contribution plus uncovered background contribution. |

## Visual storyboard

1. **Raster device array map**: One synthetic scene is interpreted as a display subpixel pattern, a binary halftone pattern, and a Bayer-style sensor mosaic. Library route: NumPy and Matplotlib because the concept is a 2D array and small device patterns.
2. **Pixel coordinate sampling**: A 4 by 3 coordinate diagram and sampled function expose the half-pixel image domain convention. Library route: Matplotlib because labeled coordinate geometry is the point.
3. **RGB color cube**: An interactive Plotly cube shows additive RGB colors as coordinates. Library route: Plotly because rotating the cube makes the channel axes and corner colors inspectable.
4. **Gamma transfer and checker match**: Curves and a synthetic checker/gray comparison show why linear light and encoded values differ. Library route: NumPy plus Matplotlib for exact curves and generated image strips.
5. **Quantization and clipping ramp**: Synthetic ramps show the difference between losing range and losing precision. Library route: NumPy and Matplotlib because the artifact is an image-array experiment with numeric error bounds.
6. **Alpha over and premultiplied alpha**: A soft synthetic foreground over a checkerboard plus a filtered-edge counterexample show why premultiplied color is the right representation for interpolation. Library route: NumPy and Pillow/Matplotlib for controllable RGBA arrays.
7. **Applied lab**: A tiny end-to-end image pipeline records array shape, dtype, value range, quantization error, and artifact existence.

The notebook uses Matplotlib for durable labeled diagrams, Plotly for the interactive RGB cube, Pillow through the course artifact helpers for saved PNGs, and NumPy for the identities. These choices match the chapter: raster images are 2D arrays with color-vector values, not 3D meshes or topology objects.


In [ ]:
CHAPTER = 3
TITLE = "Raster Images"
TOPIC = f"chapter-{CHAPTER:02d}"
PRINTED_PAGES = "63-78"
PDF_PAGES = "80-95"


In [ ]:
from pathlib import Path
import sys

BOOK_ROOT = Path.cwd()
for candidate in [BOOK_ROOT, *BOOK_ROOT.parents]:
    if (candidate / "00-book-index.ipynb").exists() and (candidate / "utils").exists():
        BOOK_ROOT = candidate
        break
else:
    raise RuntimeError("Could not find the FCG book root")

if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))

ARTIFACT_ROOT = BOOK_ROOT / "artifacts" / TOPIC
for kind in ["figures", "html", "checks", "tables", "data"]:
    (ARTIFACT_ROOT / kind).mkdir(parents=True, exist_ok=True)


In [ ]:
import json
import math

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import plotly.graph_objects as go
from IPython.display import display

from utils.artifacts import (
    assert_artifacts,
    book_relative,
    display_artifact,
    save_image,
    save_json,
    save_matplotlib,
    save_plotly_html,
    save_table_csv,
)
from utils.notebook_checks import assert_nonblank_image
from utils.plotting import PALETTE, add_note, close, style_axis
from utils.raster import alpha_over, bayer_masks, checkerboard, gamma_decode, gamma_encode, quantize

np.set_printoptions(precision=4, suppress=True)
CHECKS = {}
ARTIFACT_PATHS = []
IMAGE_PATHS = []


In [ ]:
def synthetic_rgb(width=192, height=128):
    # Return a deterministic linear-light RGB scene in [0, 1].
    y, x = np.mgrid[0:height, 0:width]
    u = x / (width - 1)
    v = y / (height - 1)
    red_blob = np.exp(-((u - 0.32) ** 2 + (v - 0.48) ** 2) / 0.035)
    blue_blob = np.exp(-((u - 0.72) ** 2 + (v - 0.42) ** 2) / 0.055)
    stripes = 0.5 + 0.5 * np.sin(2 * np.pi * (5.0 * u + 1.5 * v))
    rgb = np.stack(
        [
            0.10 + 0.78 * red_blob + 0.10 * u,
            0.14 + 0.62 * stripes * (1.0 - 0.25 * red_blob),
            0.12 + 0.75 * blue_blob + 0.15 * (1.0 - v),
        ],
        axis=-1,
    )
    return np.clip(rgb, 0.0, 1.0)


def luminance(rgb):
    rgb = np.asarray(rgb, dtype=float)
    return 0.2126 * rgb[..., 0] + 0.7152 * rgb[..., 1] + 0.0722 * rgb[..., 2]


def ordered_dither(gray):
    bayer4 = np.array(
        [
            [0, 8, 2, 10],
            [12, 4, 14, 6],
            [3, 11, 1, 9],
            [15, 7, 13, 5],
        ],
        dtype=float,
    )
    threshold = (bayer4 + 0.5) / 16.0
    tiled = np.tile(threshold, (math.ceil(gray.shape[0] / 4), math.ceil(gray.shape[1] / 4)))
    return (gray >= tiled[: gray.shape[0], : gray.shape[1]]).astype(float)


def rgb_subpixel_stripes(rgb, scale=10):
    h, w, _ = rgb.shape
    panel = np.ones((h * scale, w * scale * 3, 3), dtype=float)
    for j in range(h):
        for i in range(w):
            y0, y1 = j * scale, (j + 1) * scale
            x0 = i * scale * 3
            panel[y0:y1, x0 : x0 + scale, :] = [rgb[j, i, 0], 0.0, 0.0]
            panel[y0:y1, x0 + scale : x0 + 2 * scale, :] = [0.0, rgb[j, i, 1], 0.0]
            panel[y0:y1, x0 + 2 * scale : x0 + 3 * scale, :] = [0.0, 0.0, rgb[j, i, 2]]
    return panel


def bayer_mosaic_view(rgb):
    masks = bayer_masks(rgb.shape[0], rgb.shape[1])
    mosaic = np.zeros_like(rgb)
    mosaic[..., 0][masks["red"]] = rgb[..., 0][masks["red"]]
    mosaic[..., 1][masks["green"]] = rgb[..., 1][masks["green"]]
    mosaic[..., 2][masks["blue"]] = rgb[..., 2][masks["blue"]]
    return mosaic, masks


def assert_unit_interval(name, array):
    arr = np.asarray(array)
    assert np.nanmin(arr) >= -1e-12, f"{name} has values below zero"
    assert np.nanmax(arr) <= 1 + 1e-12, f"{name} has values above one"
    return {"min": float(np.nanmin(arr)), "max": float(np.nanmax(arr))}


def rgb_css(rgb):
    r, g, b = np.clip(np.asarray(rgb) * 255, 0, 255).astype(int)
    return f"rgb({r},{g},{b})"


## Raster Devices: One Array, Several Physical Interpretations

A raster image in memory is not yet a monitor, a printer, or a camera. It is a device-independent grid of samples. Different devices interpret that grid in different ways: an emissive or transmissive display may split each pixel into red, green, and blue subpixels; a simple binary printer may approximate a continuous tone by changing the density of black dots; a common camera sensor may measure only one color component at a cell and leave the other components for later reconstruction.

The next artifact uses one synthetic scene and shows three device-facing interpretations. The important inspection target is that the shape of the array stays fixed while the physical pattern changes. That is why graphics code should keep track of the array contract explicitly: height, width, channels, value range, and whether values are complete RGB triples or raw measurements.


In [ ]:
scene = synthetic_rgb(192, 128)
zoom = scene[42:54, 70:86]
subpixels = rgb_subpixel_stripes(zoom, scale=7)
gray = luminance(scene)
halftone = ordered_dither(gray)
halftone_rgb = np.repeat(halftone[..., None], 3, axis=2)
bayer_view, masks = bayer_mosaic_view(scene)

fig, axs = plt.subplots(2, 2, figsize=(11, 7.5), constrained_layout=True)
axs = axs.ravel()
axs[0].imshow(scene, origin="lower")
axs[0].set_title("Device-independent RGB array")
axs[0].set_xlabel("column i")
axs[0].set_ylabel("row j")
axs[0].add_patch(Rectangle((70, 42), 16, 12, fill=False, edgecolor="white", linewidth=1.8))

axs[1].imshow(subpixels, origin="lower", interpolation="nearest")
axs[1].set_title("Display interpretation: RGB subpixels")
axs[1].set_xlabel("three emitters per displayed pixel")
axs[1].set_ylabel("zoomed rows")

axs[2].imshow(halftone_rgb, cmap="gray", origin="lower", interpolation="nearest")
axs[2].set_title("Binary hardcopy interpretation: dot or no dot")
axs[2].set_xlabel("dot column")
axs[2].set_ylabel("dot row")

axs[3].imshow(bayer_view, origin="lower", interpolation="nearest")
axs[3].set_title("Input interpretation: Bayer-style samples")
axs[3].set_xlabel("one measured color component per cell")
axs[3].set_ylabel("sensor row")

for ax in axs:
    ax.set_xticks([])
    ax.set_yticks([])

add_note(axs[0], "same H x W x 3 synthetic array")
add_note(axs[1], "pixel value drives red, green, blue emitters")
add_note(axs[2], "continuous tone approximated by spatial density")
add_note(axs[3], "green samples are twice as common as red or blue")

raster_device_path = save_matplotlib(fig, TOPIC, "raster-device-array-map.png", dpi=180)
close(fig)
ARTIFACT_PATHS.append(raster_device_path)
IMAGE_PATHS.append(raster_device_path)

ten_mp = 10_000_000
storage_rows = [
    {"format": "1-bit grayscale mask", "channels": 1, "bits_per_channel": 1, "bits_per_pixel": 1},
    {"format": "8-bit RGB", "channels": 3, "bits_per_channel": 8, "bits_per_pixel": 24},
    {"format": "10-bit RGB", "channels": 3, "bits_per_channel": 10, "bits_per_pixel": 30},
    {"format": "14-bit raw-style RGB", "channels": 3, "bits_per_channel": 14, "bits_per_pixel": 42},
    {"format": "16-bit RGB", "channels": 3, "bits_per_channel": 16, "bits_per_pixel": 48},
    {"format": "16-bit grayscale", "channels": 1, "bits_per_channel": 16, "bits_per_pixel": 16},
    {"format": "half-float RGB", "channels": 3, "bits_per_channel": 16, "bits_per_pixel": 48},
    {"format": "32-bit float RGB", "channels": 3, "bits_per_channel": 32, "bits_per_pixel": 96},
]
for row in storage_rows:
    raw_bytes = ten_mp * row["bits_per_pixel"] / 8
    row["bytes_for_10mp"] = int(raw_bytes)
    row["mib_for_10mp"] = round(raw_bytes / (1024 ** 2), 2)

storage_table_path = save_table_csv(storage_rows, TOPIC, "pixel-format-storage.csv")
ARTIFACT_PATHS.append(storage_table_path)

mask_counts = {name: int(mask.sum()) for name, mask in masks.items()}
CHECKS["raster_devices"] = {
    "scene_shape": list(scene.shape),
    "scene_range": assert_unit_interval("scene", scene),
    "halftone_values": sorted(float(v) for v in np.unique(halftone)),
    "bayer_sample_counts": mask_counts,
    "ten_megapixel_float_rgb_mib": float(round(ten_mp * 3 * 4 / (1024 ** 2), 2)),
}

assert CHECKS["raster_devices"]["halftone_values"] == [0.0, 1.0]
assert mask_counts["green"] == mask_counts["red"] + mask_counts["blue"]
display_artifact(raster_device_path, width=880)
display(pd.DataFrame(storage_rows))
display_artifact(storage_table_path)


## Pixels and Sampling Geometry

The chapter's coordinate convention is deliberately simple: pixel sample points sit at integer coordinates. In a raster with `nx` columns and `ny` rows, the sample centers run from `(0, 0)` to `(nx - 1, ny - 1)`, and the rectangular image domain extends half a pixel beyond those centers. For a 4 by 3 image, the domain is `[-0.5, 3.5] x [-0.5, 2.5]`.

This convention matters later when camera projections, viewport transforms, texture lookup, and reconstruction filters enter the course. It also prevents a common misconception: a pixel value is a local sample or small-area average of an underlying image function. It is not the geometry itself. The rectangles in the diagram are neighborhoods attached to sample centers; the array samples a function over a domain.


In [ ]:
nx, ny = 4, 3
xs = np.arange(nx)
ys = np.arange(ny)
xx, yy = np.meshgrid(xs, ys)

def scalar_function(x, y):
    return 0.45 + 0.25 * np.sin(1.7 * x + 0.4) + 0.20 * np.cos(2.1 * y - 0.3)

fine_x = np.linspace(-0.5, nx - 0.5, 240)
fine_y = np.linspace(-0.5, ny - 0.5, 180)
fx, fy = np.meshgrid(fine_x, fine_y)
field = np.clip(scalar_function(fx, fy), 0.0, 1.0)
samples = np.clip(scalar_function(xx, yy), 0.0, 1.0)

fig, axs = plt.subplots(1, 2, figsize=(11, 4.4), constrained_layout=True)
ax = axs[0]
ax.set_xlim(-0.75, nx - 0.25)
ax.set_ylim(-0.75, ny - 0.25)
for j in ys:
    for i in xs:
        value = samples[j, i]
        ax.add_patch(Rectangle((i - 0.5, j - 0.5), 1, 1, facecolor=plt.cm.viridis(value), edgecolor="white", linewidth=1.2))
        ax.plot(i, j, "o", color="black", markersize=4)
        ax.text(i, j + 0.22, f"({i},{j})", ha="center", va="bottom", fontsize=8, color="black")
ax.add_patch(Rectangle((-0.5, -0.5), nx, ny, fill=False, edgecolor=PALETTE["red"], linewidth=2.0))
style_axis(ax, "Integer sample centers and half-pixel boundary", equal=True, xlabel="x / column coordinate", ylabel="y / row coordinate")
ax.set_xticks(np.arange(-0.5, nx, 0.5))
ax.set_yticks(np.arange(-0.5, ny, 0.5))
add_note(ax, "domain: [-0.5, nx-0.5] x [-0.5, ny-0.5]")

ax = axs[1]
ax.imshow(field, origin="lower", extent=(-0.5, nx - 0.5, -0.5, ny - 0.5), cmap="viridis", vmin=0, vmax=1)
ax.scatter(xx.ravel(), yy.ravel(), c=samples.ravel(), cmap="viridis", edgecolor="white", linewidth=1.0, s=90, vmin=0, vmax=1)
for j in ys:
    for i in xs:
        ax.text(i, j, f"{samples[j, i]:.2f}", ha="center", va="center", fontsize=8, color="white")
style_axis(ax, "Samples of a continuous image function", equal=True, xlabel="x", ylabel="y")
add_note(ax, "the array records local values, not all points")

pixel_coordinate_path = save_matplotlib(fig, TOPIC, "pixel-coordinate-sampling.png", dpi=180)
close(fig)
ARTIFACT_PATHS.append(pixel_coordinate_path)
IMAGE_PATHS.append(pixel_coordinate_path)

CHECKS["pixel_coordinates"] = {
    "nx": nx,
    "ny": ny,
    "domain_x": [-0.5, float(nx - 0.5)],
    "domain_y": [-0.5, float(ny - 0.5)],
    "sample_shape": list(samples.shape),
    "bottom_left_center": [0, 0],
    "top_right_center": [nx - 1, ny - 1],
    "sample_range": assert_unit_interval("samples", samples),
}
assert samples.shape == (ny, nx)
assert CHECKS["pixel_coordinates"]["domain_x"] == [-0.5, 3.5]
assert CHECKS["pixel_coordinates"]["domain_y"] == [-0.5, 2.5]
display_artifact(pixel_coordinate_path, width=880)


## RGB Color as a Unit Cube

For most graphics programming, RGB is an operational color space: store three channel values, send those values to independent red, green, and blue controls, and let additive mixing happen at the display. The corners of the cube are the familiar primaries and their sums. This is not a full theory of human color perception, which the course treats later. Here the key computational claim is simpler: an RGB pixel value is a 3-vector, usually with each component in `[0, 1]` before encoding and quantization.

Rotate the HTML cube and inspect the axes. The black corner has no emitted primary light. Moving along one axis turns on one primary. Moving along two axes reaches yellow, cyan, or magenta. Moving to `(1, 1, 1)` turns on all three primaries.


In [ ]:
levels = np.linspace(0, 1, 6)
R, G, B = np.meshgrid(levels, levels, levels, indexing="ij")
points = np.column_stack([R.ravel(), G.ravel(), B.ravel()])
colors = [rgb_css(p) for p in points]

corner_labels = {
    "black": (0, 0, 0),
    "red": (1, 0, 0),
    "green": (0, 1, 0),
    "blue": (0, 0, 1),
    "yellow": (1, 1, 0),
    "cyan": (0, 1, 1),
    "magenta": (1, 0, 1),
    "white": (1, 1, 1),
}
fig = go.Figure()
fig.add_trace(
    go.Scatter3d(
        x=points[:, 0],
        y=points[:, 1],
        z=points[:, 2],
        mode="markers",
        marker={"size": 5, "color": colors, "line": {"width": 0.5, "color": "#1f2933"}},
        hovertemplate="R=%{x:.2f}<br>G=%{y:.2f}<br>B=%{z:.2f}<extra></extra>",
        name="RGB samples",
    )
)
fig.add_trace(
    go.Scatter3d(
        x=[v[0] for v in corner_labels.values()],
        y=[v[1] for v in corner_labels.values()],
        z=[v[2] for v in corner_labels.values()],
        mode="text",
        text=list(corner_labels.keys()),
        textposition="top center",
        showlegend=False,
    )
)
edges = []
for a in [0, 1]:
    for b in [0, 1]:
        edges.extend([([0, 1], [a, a], [b, b]), ([a, a], [0, 1], [b, b]), ([a, a], [b, b], [0, 1])])
for x_edge, y_edge, z_edge in edges:
    fig.add_trace(go.Scatter3d(x=x_edge, y=y_edge, z=z_edge, mode="lines", line={"color": "#59636e", "width": 3}, showlegend=False))
fig.update_layout(
    title="RGB colors are coordinates in the unit cube",
    scene={
        "xaxis": {"title": "red", "range": [0, 1]},
        "yaxis": {"title": "green", "range": [0, 1]},
        "zaxis": {"title": "blue", "range": [0, 1]},
        "aspectmode": "cube",
    },
    margin={"l": 0, "r": 0, "t": 40, "b": 0},
)

rgb_cube_path = save_plotly_html(fig, TOPIC, "rgb-color-cube.html")
ARTIFACT_PATHS.append(rgb_cube_path)

CHECKS["rgb_cube"] = {
    "grid_point_count": int(points.shape[0]),
    "corner_count": len(corner_labels),
    "corner_coordinates": {name: list(map(float, value)) for name, value in corner_labels.items()},
    "all_points_in_unit_cube": bool(np.all((points >= 0) & (points <= 1))),
}
assert CHECKS["rgb_cube"]["grid_point_count"] == 216
assert CHECKS["rgb_cube"]["corner_coordinates"]["white"] == [1.0, 1.0, 1.0]
assert CHECKS["rgb_cube"]["all_points_in_unit_cube"]
display_artifact(rgb_cube_path, width="100%", height=560)


## Gamma: Stored Codes Are Not Linear Light

A rendering calculation usually wants linear light: doubling a value should double the physical intensity. A typical display transfer curve is nonlinear, often approximated by `displayed_intensity = code_value ** gamma`. With `gamma = 2.2`, a code value of `0.5` emits much less than half the maximum intensity. To ask a display for half intensity, the code must be about `0.5 ** (1 / 2.2)`.

The checker comparison is useful because alternating black and white pixels blur to half average intensity when viewed at a distance. Matching a solid gray patch against that blurred checker estimates the display transfer. The artifact below separates three spaces that often get confused: intended linear intensity, encoded code value, and predicted displayed intensity.


In [ ]:
gamma = 2.2
code = np.linspace(0, 1, 256)
linear = np.linspace(0, 1, 256)
a_mid = float(0.5 ** (1.0 / gamma))
encoded_for_linear = gamma_encode(linear, gamma)
roundtrip = gamma_decode(encoded_for_linear, gamma)

check_size = 160
checker = ((np.indices((check_size, check_size)).sum(axis=0) % 2) == 0).astype(float)
encoded_match = np.ones((check_size, check_size * 2), dtype=float)
encoded_match[:, :check_size] = checker
encoded_match[:, check_size:] = a_mid
predicted_intensity = encoded_match ** gamma

fig, axs = plt.subplots(2, 2, figsize=(11, 7.5), constrained_layout=True)
ax = axs[0, 0]
for g, color in [(1.0, PALETTE["gray"]), (1.8, PALETTE["teal"]), (2.2, PALETTE["blue"]), (2.6, PALETTE["red"])]:
    ax.plot(code, code ** g, label=f"gamma={g:g}", color=color)
ax.axhline(0.5, color="#4b5563", linestyle="--", linewidth=1)
ax.axvline(a_mid, color=PALETTE["red"], linestyle="--", linewidth=1)
style_axis(ax, "Display transfer: code -> emitted intensity", xlabel="stored code value a", ylabel="relative emitted intensity")
ax.legend(frameon=False)
add_note(ax, f"half intensity at code {a_mid:.3f} for gamma={gamma}")

ax = axs[0, 1]
ax.plot(linear, encoded_for_linear, color=PALETTE["green"], label="encode linear intensity")
ax.plot(linear, linear, color=PALETTE["gray"], linestyle="--", label="linear reference")
style_axis(ax, "Gamma correction: linear target -> stored code", xlabel="intended linear intensity", ylabel="stored code value")
ax.legend(frameon=False)
add_note(ax, "compute in linear space, encode at the boundary")

ax = axs[1, 0]
ax.imshow(encoded_match, cmap="gray", vmin=0, vmax=1, interpolation="nearest")
ax.set_title("Encoded values sent to display")
ax.set_xticks([])
ax.set_yticks([])
add_note(ax, "left: 0/1 checker, right: solid code a")

ax = axs[1, 1]
ax.imshow(predicted_intensity, cmap="gray", vmin=0, vmax=1, interpolation="nearest")
ax.set_title("Predicted emitted intensity after gamma")
ax.set_xticks([])
ax.set_yticks([])
add_note(ax, "solid patch and checker average both target 0.5")

gamma_path = save_matplotlib(fig, TOPIC, "gamma-transfer-checker.png", dpi=180)
close(fig)
ARTIFACT_PATHS.append(gamma_path)
IMAGE_PATHS.append(gamma_path)

CHECKS["gamma"] = {
    "gamma": gamma,
    "code_for_half_intensity": a_mid,
    "half_intensity_residual": float(abs(a_mid ** gamma - 0.5)),
    "roundtrip_max_error": float(np.max(np.abs(roundtrip - linear))),
    "encoded_monotone": bool(np.all(np.diff(encoded_for_linear) >= -1e-12)),
}
assert CHECKS["gamma"]["half_intensity_residual"] < 1e-12
assert CHECKS["gamma"]["roundtrip_max_error"] < 1e-12
assert CHECKS["gamma"]["encoded_monotone"]
display_artifact(gamma_path, width=880)


## Quantization and Clipping

Two different losses happen when a rich image is forced into a finite fixed-range format. **Clipping** loses values outside the allowed interval, so any highlight above 1.0 becomes indistinguishable from exactly 1.0. **Quantization** keeps the range but rounds to a finite set of representable values. A one-bit image has only black and white. An eight-bit channel has 256 representable levels, including both 0 and 1.

The artifact below uses a smooth ramp so the errors are easy to see and easy to assert. Banding is the visual version of quantization error. A check can bound that error by half a quantization step for values already inside the range.


In [ ]:
width = 512
height = 34
ramp = np.linspace(0, 1, width)
ramp_img = np.repeat(ramp[None, :, None], height, axis=0).repeat(3, axis=2)
hdr_ramp = np.linspace(0, 1.6, width)
hdr_img = np.repeat(np.clip(hdr_ramp[None, :, None], 0, 1), height, axis=0).repeat(3, axis=2)

bit_depths = [1, 2, 4, 8]
quantized_rows = []
quantization_checks = {}
for bits in bit_depths:
    levels_count = 2 ** bits
    q = quantize(ramp_img, levels_count)
    quantized_rows.append(q)
    err = np.abs(q[..., 0] - ramp_img[..., 0])
    step = 1.0 / (levels_count - 1)
    quantization_checks[str(bits)] = {
        "levels": levels_count,
        "unique_values": int(np.unique(q[..., 0]).size),
        "max_abs_error": float(err.max()),
        "half_step_bound": float(step / 2),
    }

fig = plt.figure(figsize=(11, 7.2), constrained_layout=True)
gs = fig.add_gridspec(3, 2, height_ratios=[1.15, 1.15, 1.0])
ax = fig.add_subplot(gs[0, 0])
ax.imshow(np.vstack([ramp_img, hdr_img]), origin="lower", vmin=0, vmax=1)
ax.set_title("Range loss: HDR ramp clipped to [0, 1]")
ax.set_xticks([])
ax.set_yticks([height / 2, height + height / 2])
ax.set_yticklabels(["0..1 ramp", "0..1.6 clipped"])
add_note(ax, "values above 1 collapse to display white")

ax = fig.add_subplot(gs[0, 1])
ax.plot(hdr_ramp, np.clip(hdr_ramp, 0, 1), color=PALETTE["red"])
style_axis(ax, "Clipping transfer", xlabel="input linear value", ylabel="stored value")
ax.axvline(1.0, color="#4b5563", linestyle="--", linewidth=1)

ax = fig.add_subplot(gs[1, :])
stacked = np.vstack(quantized_rows)
ax.imshow(stacked, origin="lower", vmin=0, vmax=1, interpolation="nearest")
ax.set_title("Precision loss: quantized ramps")
ax.set_xticks([])
row_centers = [height * (idx + 0.5) for idx in range(len(bit_depths))]
ax.set_yticks(row_centers)
ax.set_yticklabels([f"{bits}-bit" for bits in bit_depths])
add_note(ax, "banding shrinks as channel precision increases")

ax = fig.add_subplot(gs[2, :])
for bits in [2, 4, 8]:
    levels_count = 2 ** bits
    q = quantize(ramp, levels_count)
    ax.plot(ramp, q - ramp, label=f"{bits}-bit error")
style_axis(ax, "Quantization error", xlabel="linear value", ylabel="stored - original")
ax.legend(frameon=False, ncols=3)

quantization_path = save_matplotlib(fig, TOPIC, "quantization-clipping-ramp.png", dpi=180)
close(fig)
ARTIFACT_PATHS.append(quantization_path)
IMAGE_PATHS.append(quantization_path)

CHECKS["quantization"] = quantization_checks
for bits, item in quantization_checks.items():
    assert item["unique_values"] <= item["levels"]
    assert item["max_abs_error"] <= item["half_step_bound"] + 1e-12
assert quantization_checks["8"]["levels"] == 256
display_artifact(quantization_path, width=880)


## Alpha Compositing and Premultiplied Alpha

Alpha is the fraction of a pixel covered by a foreground layer, or more generally the opacity of a layer. With an opaque background, the basic over operation is

`out_rgb = alpha * foreground_rgb + (1 - alpha) * background_rgb`.

That equation is simple, but storage details matter. In a straight-alpha RGBA image, RGB and alpha are stored separately. Fully transparent pixels can still contain arbitrary RGB values, which becomes dangerous when the image is filtered, resampled, or interpolated. Premultiplied alpha stores `alpha * RGB` instead. Filtering premultiplied color keeps invisible colors from leaking into visible edges.

The right side of the artifact is a small counterexample: interpolate from opaque red to transparent blue. Straight-alpha interpolation creates a blue fringe after compositing over white because the hidden blue RGB value was averaged before alpha was applied. Premultiplied interpolation does not leak the hidden color.


In [ ]:
size = 256
bg = checkerboard(size, size, checks=10)
y, x = np.mgrid[0:size, 0:size]
u = (x + 0.5) / size
v = (y + 0.5) / size
radius = np.sqrt((u - 0.52) ** 2 + (v - 0.50) ** 2)
alpha = np.clip((0.34 - radius) / 0.08, 0.0, 1.0)
fg = np.stack(
    [
        0.95 - 0.35 * v,
        0.16 + 0.48 * u,
        0.08 + 0.82 * (1 - u) * v,
    ],
    axis=-1,
)
fg = np.clip(fg, 0.0, 1.0)
composite = alpha_over(fg, bg, alpha)
premul = fg * alpha[..., None]
premul_composite = premul + bg * (1 - alpha[..., None])

edge_width = 360
edge_height = 58
t = np.linspace(0, 1, edge_width)
left_rgb = np.array([1.0, 0.05, 0.0])
right_hidden_rgb = np.array([0.0, 0.1, 1.0])
straight_rgb = (1 - t)[:, None] * left_rgb + t[:, None] * right_hidden_rgb
edge_alpha = 1 - t
straight_over_white = straight_rgb * edge_alpha[:, None] + (1 - edge_alpha[:, None]) * 1.0
premul_rgb = (1 - t)[:, None] * left_rgb + t[:, None] * (right_hidden_rgb * 0.0)
premul_over_white = premul_rgb + (1 - edge_alpha[:, None]) * 1.0
straight_strip = np.repeat(straight_over_white[None, :, :], edge_height, axis=0)
premul_strip = np.repeat(premul_over_white[None, :, :], edge_height, axis=0)

fig, axs = plt.subplots(2, 3, figsize=(11, 7.5), constrained_layout=True)
axs[0, 0].imshow(fg, origin="lower")
axs[0, 0].set_title("Foreground RGB")
axs[0, 1].imshow(alpha, cmap="gray", origin="lower", vmin=0, vmax=1)
axs[0, 1].set_title("Alpha coverage")
axs[0, 2].imshow(composite, origin="lower")
axs[0, 2].set_title("Over checkerboard")
axs[1, 0].imshow(premul, origin="lower")
axs[1, 0].set_title("Premultiplied color alpha*RGB")
axs[1, 1].imshow(straight_strip, origin="lower", vmin=0, vmax=1)
axs[1, 1].set_title("Straight-alpha filtering: hidden blue leaks")
axs[1, 2].imshow(premul_strip, origin="lower", vmin=0, vmax=1)
axs[1, 2].set_title("Premultiplied filtering: no hidden color")
for ax in axs.ravel():
    ax.set_xticks([])
    ax.set_yticks([])
add_note(axs[0, 2], "out = alpha*fg + (1-alpha)*bg")
add_note(axs[1, 1], "RGB averaged before alpha")
add_note(axs[1, 2], "alpha*RGB averaged directly")

alpha_path = save_matplotlib(fig, TOPIC, "alpha-over-premultiplied.png", dpi=180)
close(fig)
ARTIFACT_PATHS.append(alpha_path)
IMAGE_PATHS.append(alpha_path)

fringe_index = edge_width // 2
CHECKS["alpha"] = {
    "alpha_range": assert_unit_interval("alpha", alpha),
    "straight_premul_equivalence_max_error": float(np.max(np.abs(composite - premul_composite))),
    "straight_mid_edge_rgb": [float(v) for v in straight_over_white[fringe_index]],
    "premultiplied_mid_edge_rgb": [float(v) for v in premul_over_white[fringe_index]],
    "hidden_blue_leak": float(straight_over_white[fringe_index, 2] - premul_over_white[fringe_index, 2]),
}
assert CHECKS["alpha"]["straight_premul_equivalence_max_error"] < 1e-12
assert CHECKS["alpha"]["hidden_blue_leak"] > 0.10
assert np.all(premul <= alpha[..., None] + 1e-12)
display_artifact(alpha_path, width=880)


## Applied lab: Audit an Image Array Before Saving It

A practical raster pipeline should be able to answer boring questions before anyone argues about aesthetics: What is the array shape? Which axis is rows? What is the channel count? Are the values linear or encoded? Is the final file integer or floating point? How much error did quantization introduce?

The lab below builds a tiny synthetic image from linear RGB data, composites a premultiplied badge in linear light, gamma-encodes the result, quantizes to 8-bit storage, and records the array facts. The resulting PNG is not meant to be a beautiful picture. It is a checksum-friendly artifact that proves the chapter's mechanics can be executed end to end.


In [ ]:
lab_linear = synthetic_rgb(160, 96)
lab_h, lab_w, _ = lab_linear.shape
lab_y, lab_x = np.mgrid[0:lab_h, 0:lab_w]
lab_u = (lab_x + 0.5) / lab_w
lab_v = (lab_y + 0.5) / lab_h
badge_alpha = np.clip((0.23 - np.sqrt((lab_u - 0.72) ** 2 + (lab_v - 0.30) ** 2)) / 0.05, 0.0, 1.0)
badge_rgb = np.zeros_like(lab_linear)
badge_rgb[..., 0] = 1.0
badge_rgb[..., 1] = 0.82
badge_rgb[..., 2] = 0.05
lab_composite_linear = badge_rgb * badge_alpha[..., None] + lab_linear * (1 - badge_alpha[..., None])
lab_encoded = gamma_encode(lab_composite_linear, gamma)
lab_u8 = np.round(np.clip(lab_encoded, 0, 1) * 255).astype(np.uint8)
lab_decoded = gamma_decode(lab_u8.astype(float) / 255.0, gamma)
lab_quantization_error = np.abs(lab_decoded - lab_composite_linear)

lab_image_path = save_image(lab_u8, TOPIC, "image-array-lab-output.png")
ARTIFACT_PATHS.append(lab_image_path)
IMAGE_PATHS.append(lab_image_path)

lab_summary = {
    "linear_shape": list(lab_linear.shape),
    "saved_dtype": str(lab_u8.dtype),
    "saved_shape": list(lab_u8.shape),
    "encoded_min": float(lab_encoded.min()),
    "encoded_max": float(lab_encoded.max()),
    "badge_alpha_min": float(badge_alpha.min()),
    "badge_alpha_max": float(badge_alpha.max()),
    "decoded_quantization_max_error_linear": float(lab_quantization_error.max()),
    "bytes_if_float32_rgb": int(lab_w * lab_h * 3 * 4),
    "bytes_if_uint8_rgb": int(lab_w * lab_h * 3),
}
lab_summary_path = save_json(lab_summary, TOPIC, "image-array-lab-summary.json")
ARTIFACT_PATHS.append(lab_summary_path)
CHECKS["applied_lab"] = lab_summary

assert lab_u8.dtype == np.uint8
assert lab_u8.shape == lab_linear.shape
assert lab_summary["encoded_min"] >= 0.0 and lab_summary["encoded_max"] <= 1.0
assert lab_summary["bytes_if_float32_rgb"] == 4 * lab_summary["bytes_if_uint8_rgb"]
display_artifact(lab_image_path, width=640)
display_artifact(lab_summary_path)
lab_summary


## Sanity checks

The checks below are part of the lesson. They assert the identities that make the visuals meaningful instead of decorative:

- The device-map image is nonblank and the binary printer pattern contains only two values.
- The pixel coordinate diagram uses the stated half-pixel domain.
- The RGB cube contains all expected corner colors inside the unit cube.
- Gamma encoding and decoding round trip in floating point, and the half-intensity code produces half intensity under the transfer curve.
- Quantization error stays within half a storage step.
- Straight alpha over an opaque background equals premultiplied color plus uncovered background, while straight-alpha interpolation leaks hidden color.
- The lab output has the expected shape and storage ratio.


In [ ]:
invariant_path = save_json(CHECKS, TOPIC, "raster-image-invariants.json")
ARTIFACT_PATHS.append(invariant_path)

artifact_records = assert_artifacts(ARTIFACT_PATHS)
raw_image_records = [assert_nonblank_image(path) for path in IMAGE_PATHS]
image_records = [{**record, "path": book_relative(record["path"])} for record in raw_image_records]

assert CHECKS["raster_devices"]["scene_shape"] == [128, 192, 3]
assert CHECKS["pixel_coordinates"]["sample_shape"] == [3, 4]
assert CHECKS["rgb_cube"]["corner_coordinates"]["yellow"] == [1.0, 1.0, 0.0]
assert CHECKS["gamma"]["roundtrip_max_error"] < 1e-12
assert CHECKS["quantization"]["8"]["levels"] == 256
assert CHECKS["alpha"]["straight_premul_equivalence_max_error"] < 1e-12
assert CHECKS["applied_lab"]["saved_dtype"] == "uint8"

final_report = {
    "chapter": CHAPTER,
    "title": TITLE,
    "printed_pages": PRINTED_PAGES,
    "pdf_pages": PDF_PAGES,
    "artifact_count": len(artifact_records),
    "nonblank_image_count": len(image_records),
    "artifact_records": artifact_records,
    "image_records": image_records,
    "check_sections": sorted(CHECKS),
}
final_path = save_json(final_report, TOPIC, "final-sanity.json")
assert_artifacts([final_path])
display_artifact(invariant_path)
display_artifact(final_path)
final_report


## Takeaways

A raster image is a sampled, finite representation of an underlying image function. Real devices explain why this representation is useful, but the array is not tied to one device. The same values can become display subpixels, printer dots, sensor samples, file bytes, or intermediate linear-light computations.

The most important implementation habit is to name the space a value lives in. Pixel coordinates need a convention. RGB values need channel order and range. Display-bound values need a transfer curve. Fixed-range formats need clipping and quantization checks. Alpha-bearing images need a clear distinction between straight and premultiplied color.

Later chapters will add camera transforms, rasterization, filtering, textures, shading, and perception. The chapter's core invariant will stay the same: before trusting a picture, inspect the array and check the math that produced it.
